In [1]:
from src.gaussian_mixture import GaussianMixtureMAR
from src.motivating_example import MotivatingExample
from src.data_generator import DataGenerator
import numpy as np

In [ ]:
import numpy as np
import torch
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

mice = importr("mice")


class AmputeMAR:

    def __init__(self, d=None, n=None, seed=None):
        self.rng = np.random.default_rng(seed)
        self.patterns = None
        if d is not None and n is not None:
            self.patterns = self.sample_patterns(d, n, self.rng)
    
    def _to_r_dataframe(self, X):
        return ro.DataFrame({
            f"X{j + 1}": ro.FloatVector(X[:, j])
            for j in range(X.shape[1])
        })

    def _to_r_matrix(self, X):
        return ro.r.matrix(
            ro.IntVector(X.T.flatten().tolist()),
            nrow=X.shape[0],
            ncol=X.shape[1],
        )

    def sample_patterns(self, d, n, rng=None):
        """Sample incomplete missingness patterns once (1 = observed, 0 = missing).
        Last column is always observed; all-ones rows are excluded."""
        rng = np.random.default_rng() if rng is None else rng
        while True:  # resample in the (rare) case no incomplete pattern is drawn
            npattern = int(rng.integers(3, max(round(n / 100), 3) + 1))
            patterns = rng.integers(0, 2, size=(npattern, d - 1))
            patterns = np.hstack([patterns, np.ones((npattern, 1), dtype=int)])
            incomplete = patterns[~patterns.all(axis=1)]
            if incomplete.shape[0] > 0:
                return incomplete


    def ampute(self, X, patterns=None, prop=0.3, keep_complete=True, rng=None):
        """
        Parameters
        ----------
        X : torch.Tensor, shape (n, d)
            Complete data.
        patterns : array-like, shape (n_pats, d), optional
            Binary patterns: 1 = observed, 0 = missing.
            If None, uses self.patterns (sampled once and cached).
            All-ones rows are dropped and handled via `keep_complete`.
        prop : float
            Proportion of eligible cases to ampute.
        keep_complete : bool
            If True, hold out round(n / (n_pats + 1)) random rows as fully
            observed, and ampute only the remaining rows.
        rng : np.random.Generator, optional
            Defaults to self.rng.

        Returns
        -------
        X_NA : torch.Tensor
            Data with NaN values.
        M : torch.Tensor
            Observation mask: 1 = observed, 0 = missing.
        """
        rng = self.rng if rng is None else rng          # changed
        n, d = X.shape
        device = X.device
        X_np = X.detach().cpu().numpy().astype(float)

        if patterns is None:                            # changed
            if self.patterns is None:
                self.patterns = self.sample_patterns(d, n, rng)
            patterns = self.patterns
        patterns = np.asarray(patterns, dtype=int)

        # Separate the all-ones (complete) pattern
        patterns = patterns[~patterns.all(axis=1)]
        n_pats = patterns.shape[0]

        X_NA_np = X_np.copy()

        if n_pats > 0:
            # Randomly assign rows to the "complete" pattern
            if keep_complete:
                n_complete = int(round(n / (n_pats + 1)))
                complete_idx = rng.choice(n, size=n_complete, replace=False)
            else:
                complete_idx = np.array([], dtype=int)

            mask = np.ones(n, dtype=bool)
            mask[complete_idx] = False
            rest_idx = np.flatnonzero(mask)

            # Ampute the remaining rows (equal pattern frequencies = mice default)
            #X_rest = X[torch.as_tensor(rest_idx, device=device)]
            result = mice.ampute(
                self._to_r_dataframe(X_np[rest_idx]),
                patterns=self._to_r_matrix(patterns),
                prop=prop,
                mech="MAR",
                bycases=True,
            )
            amp = np.asarray(result.rx2("amp")).T  # (n_rest, d)

            # Write back in place: original row order is preserved automatically
            X_NA_np[rest_idx] = amp

        M_np = (~np.isnan(X_NA_np)).astype(np.float32)

        return (
            torch.tensor(X_NA_np, dtype=X.dtype, device=device),
            torch.tensor(M_np, dtype=X.dtype, device=device),
        )

    # def ampute(self, X, patterns=None, prop=0.3):
    #     """
    #     Parameters
    #     ----------
    #     X : torch.Tensor, shape (n, d)
    #         Complete data.
    #     patterns : array-like, shape (n, d)
    #         Binary patterns: 1 = observed, 0 = missing.
    #         All-ones rows are kept complete.
    #     prop : float
    #         Proportion of eligible cases to ampute.

    #     Returns
    #     -------
    #     X_NA : torch.Tensor
    #         Data with NaN values.
    #     M : torch.Tensor
    #         Observation mask: 1 = observed, 0 = missing.
    #     """
    #     if patterns is None:
    #         patterns = np.ones_like(X, dtype=int)
    #         n, d = X.shape
    #         patterns[:, :-1] = np.random.randint(0, 2, size=(n, d - 1))
    #         patterns[:, -1] = 1

    #     device = X.device
    #     X_np = X.detach().cpu().numpy().astype(float)
    #     patterns = np.asarray(patterns)


    #     result = mice.ampute(
    #         self._to_r_dataframe(X),
    #         patterns=self._to_r_matrix(patterns),
    #         prop=prop,
    #         mech="MAR",
    #         bycases=True,
    #     )

    #     X_NA_np = X_np.copy()
    #     X_NA_np = np.asarray(result.rx2("amp")).T

    #     M_np = (~np.isnan(X_NA_np)).astype(np.float32)

    #     return (
    #         torch.tensor(X_NA_np, dtype=X.dtype, device=device),
    #         torch.tensor(M_np, dtype=X.dtype, device=device),
    #     )

In [3]:
class ParametricEstimation(GaussianMixtureMAR, AmputeMAR):

    def __init__(self, 
                 data_generator,
                 n,
                 d,
                 patterns,
                 k_range, 
                 criterion="bic", 
                 cov_type='uni_diag', 
                 max_iter=100, 
                 tol=0.000001, 
                 eps=0.000001, 
                 reg_covar=0.000001, 
                 n_init=20, 
                 _k_tol=1, 
                 device="cpu", 
                 verbose=False):
        super().__init__(k_range, criterion, cov_type, max_iter, tol, eps, reg_covar, n_init, _k_tol, device, verbose)
        self.n = n
        self.d = d
        self.X = data_generator.sample_truth(n, d)
        self.Xna, self.M = self.ampute(self.X, patterns=patterns)
        self.model = self.fit(self.X, self.M)

    def mean(self):
        Xtilde = self.model.sample(self.n)[0]
        return Xtilde.mean(dim=0)

    def quantile(self, var_index=0, p=0.1):
        """
        Estimate the p-th quantile of the variable with index var_index
        """
        Xtilde = self.model.sample(self.n)[0]
        x = Xtilde[: var_index]
        return np.quantile(x, p)

    def estimate(self, func):
        Xtilde = self.model.sample(self.n)[0]
        return func(Xtilde)

    # def truth(self, func):
    #     """the truth not from here, need to know data generating mechanism"""
    #     pass

In [4]:
dg = DataGenerator(distr='Logistic')

In [5]:
n = 2000
d = 3
B = 2

In [6]:
means = torch.empty((B, d))
quantiles_X2 = torch.empty(B)

for i in range(2):
    pe = ParametricEstimation(data_generator=dg, n=n, d=d, patterns=None, k_range=(0,10))
    means[i] = pe.mean()
    quantiles_X2[i] = pe.quantile(var_index=1, p=0.2).item()

In [7]:
print('Mean estimates per batch: \n', means)

Mean estimates per batch: 
 tensor([[-0.0096, -0.0775,  0.0294],
        [-0.1122, -0.0868,  0.0380]])


In [8]:
print('Quantile 0.2 estimates of X2 per batch: \n', quantiles_X2)

Quantile 0.2 estimates of X2 per batch: 
 tensor([-0.7753, -1.2305])
